In [ ]:
import pandas as pd
import fastf1
import os

In [ ]:
#folder za kes
cache_path = 'temp_cache'
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
fastf1.set_log_level('ERROR') #da se smanji ispis fastf1 biblioteke

In [ ]:
# ucitava se csv
df = pd.read_csv('all_races.csv')
print(f"Sve trke u CSV: {len(df)}")

#uzimaju se samo prv
df = df[['season', 'round', 'race_name', 'date', 'time']]
df.columns = ['season', 'round', 'race_name', 'date', 'time']
df = df[df['season'] >= 2018]  # samo posle 2018, za pre ne postoje podaci
print(f"Trke posle 2018 : {len(df)}")

Sve trke u CSV: 149
Trke posle 2018 : 149


In [ ]:
# Ucitaj podatke koji vec postoje
processed_file = 'all_weather.csv'
if os.path.exists(processed_file):
    df_done = pd.read_csv(processed_file)
    completed_set = set(zip(df_done['season'], df_done['round']))
    print(f"Vec postojece: {len(completed_set)}")
else:
    df_done = pd.DataFrame()
    completed_set = set()
    print("Ne postoji nista")

Vec postojece: 148


In [ ]:
# CUcitavaju se novi podaci
new_rows = []

for idx, row in df.iterrows():
    season = int(row['season'])
    round_num = int(row['round'])

    if (season, round_num) in completed_set:
        print(f"Preskace se {season} runda {round_num}: vec postoji")
        continue

    try:
        print(f"Ucitavanje {season} runda {round_num}...")
        session = fastf1.get_session(season, round_num, 'R')
        session.load(telemetry=False, weather=True)

        weather_df = session.weather_data
        if weather_df.empty:
            print(f"Ne postoje podaci za vreme za {season} runda {round_num}")
            continue

        mean_weather = weather_df.mean(numeric_only=True).to_dict()
        enriched_row = row.to_dict()
        enriched_row.update(mean_weather)
        new_rows.append(enriched_row)
        print(f"Dodato vreme za {season} runda {round_num}")

    except Exception as e:
        print(f"Neupesno za {season} runda {round_num}: {e}")

Preskace se 2018 runda 1: vec postoji
Preskace se 2018 runda 2: vec postoji
Preskace se 2018 runda 3: vec postoji
Preskace se 2018 runda 4: vec postoji
Preskace se 2018 runda 5: vec postoji
Preskace se 2018 runda 6: vec postoji
Preskace se 2018 runda 7: vec postoji
Preskace se 2018 runda 8: vec postoji
Preskace se 2018 runda 9: vec postoji
Preskace se 2018 runda 10: vec postoji
Preskace se 2018 runda 11: vec postoji
Preskace se 2018 runda 12: vec postoji
Preskace se 2018 runda 13: vec postoji
Preskace se 2018 runda 14: vec postoji
Preskace se 2018 runda 15: vec postoji
Preskace se 2018 runda 16: vec postoji
Preskace se 2018 runda 17: vec postoji
Preskace se 2018 runda 18: vec postoji
Preskace se 2018 runda 19: vec postoji
Preskace se 2018 runda 20: vec postoji
Preskace se 2018 runda 21: vec postoji
Preskace se 2019 runda 1: vec postoji
Preskace se 2019 runda 2: vec postoji
Preskace se 2019 runda 3: vec postoji
Preskace se 2019 runda 4: vec postoji
Preskace se 2019 runda 5: vec postoji


Request for URL https://ergast.com/api/f1/2024/24/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connection.py", line 200, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py", line 790, in urlopen
    res

Dodato vreme za 2024 runda 24


In [ ]:
# Kombinovanje sa starim i cuvanje
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_done, df_new], ignore_index=True)
    df_combined.to_csv(processed_file, index=False)
    print(f"Sacuvano {len(df_new)} novih redova {processed_file}")
else:
    print("Nista nije dodato")


Sacuvano 1 novih redova all_weather.csv
